#대학 등록금 데이터 전처리 템플릿 (2010~2023)

In [ ]:
import pandas as pd

# 파일 경로를 수동으로 지정
file_paths = [
    "data/2010년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "data/2011년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "data/2012년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "data/2013년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "data/2014년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "data/2015년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "data/2016년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "data/2017년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "data/2018년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "data/2019년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "data/2020년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "data/2021년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "data/2022년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "data/2023년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx"
]
file_paths

In [ ]:

# 사이버대 제거 키워드
cyber_keywords = ['사이버', '방송통신']

# 2023 기준 학교명 기준으로 통일
df_2023 = pd.read_excel(file_paths[-1], header=3)  # 2023 기준
df_2023 = df_2023[~df_2023['학교'].str.contains('|'.join(cyber_keywords), na=False)]
school_names_2023 = df_2023['학교'].unique()


In [ ]:

dfs = {}
for path in file_paths:
    year = int(Path(path).stem[:4])
    df = pd.read_excel(path, header=3)
    df = df.rename(columns={'학교': '학교명'})
    df = df[df['학교명'].isin(school_names_2023)]
    df = df[~df['학교명'].str.contains('|'.join(cyber_keywords), na=False)]
    if '기성회비＊\n(C)' in df.columns:
        df = df.drop(columns=['기성회비＊\n(C)'], errors='ignore')
    if '전년도\n등록금' in df.columns:
        df = df.drop(columns=['전년도\n등록금'], errors='ignore')
    df['기준년도'] = year
    dfs[year] = df


In [ ]:

# 공통 학교 교집합
common_schools = set(school_names_2023)
for df in dfs.values():
    common_schools &= set(df['학교명'].dropna().unique())

# 필터링 및 등록금 생성
processed = []
for year, df in dfs.items():
    df = df[df['학교명'].isin(common_schools)].copy()
    df['등록금'] = df.get('등록금\n(D=B+C)', pd.NA).combine_first(df.get('등록금\n(D=B)', pd.NA))
    df['입학금\n(A)'] = df.get('입학금\n(A)', 0).fillna(0).round(1)
    df['등록금'] = df['등록금'] + df['입학금\n(A)']
    df['기준년도'] = year
    df = df.drop(columns=['입학금\n(A)', '등록금\n(D=B)', '등록금\n(D=B+C)', '학교종류', '상태'], errors='ignore')
    processed.append(df)

final_df = pd.concat(processed, ignore_index=True)
final_df.to_csv("최종_등록금_통합_데이터_2010_2023.csv", index=False, encoding="utf-8-sig")
final_df.head()
